# 02 - Validating like the holdout is real

The holdout values sit in the same file as the training data. Any evaluation that
touches them is worthless. So we build the evaluation entirely inside the 110
training points.

In [1]:
import sys, json
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np, pandas as pd
pd.set_option("display.width", 120)
ART = ROOT / "reports" / "artifacts"
print("project root:", ROOT)

project root: C:\Z_Coding\Cloud-Workload-Forecasting


## The folds

Expanding window, horizon 24, origins every 6 steps. The constraint is
`origin + horizon <= 110` for every fold.

In [2]:
from cwf.evaluate import make_folds, assert_folds_valid

folds = make_folds(110, min_train=60, step=6, horizon=24)
assert_folds_valid(folds, 110)
for f in folds:
    print(f"origin={f.origin:3d}  train={f.train_slice.stop:3d}  test={f.test_slice.start}-{f.test_slice.stop - 1}")

origin= 60  train= 61  test=61-84
origin= 66  train= 67  test=67-90
origin= 72  train= 73  test=73-96
origin= 78  train= 79  test=79-102
origin= 84  train= 85  test=85-108


Notice the last fold: origin 84, test window 85-108, still inside 110. Nothing
ever reaches into the holdout.

## Results as measured, not as hoped

The table below is the committed artifact from `scripts/run_validation.py`.

In [3]:
results = pd.read_csv(ART / "validation_results.csv")
results.sort_values(["target", "rmse"]).to_string(index=False)

'     mae     rmse      bias     mase  scale_naive_mae  mape_pct  mape_excluded_points        candidate                    target  n_folds  rmse_mean_per_origin  rmse_se_across_origins  rmse_min_per_origin  rmse_max_per_origin  within_1se_of_best  selected  complexity_rank\n0.000000 0.000000  0.000000      NaN          0.00000  0.000000                     0      persistence           active_vm_count        5              0.000000                0.000000             0.000000             0.000000                True      True              0.0\n0.000000 0.000000  0.000000      NaN          0.00000  0.000000                     0              ma6           active_vm_count        5              0.000000                0.000000             0.000000             0.000000                True     False              1.0\n0.000000 0.000000  0.000000      NaN          0.00000  0.000000                     0             ma12           active_vm_count        5              0.000000                0.

Selection rule, frozen in `docs/protocol.md` before any of this ran: lowest
rolling-origin RMSE, and if a candidate is within one standard error of the best,
prefer the simpler. Applied to the table above it selects **persistence** for both
targets.

For `active_vm_count` every candidate has RMSE 0.0 because the series does not
move. MASE is undefined (zero naive scale). Both facts are printed, not hidden.

## How noisy is this?

Five overlapping origins is not a lot of evidence. Look at the spread across
origins before believing any ranking.

In [4]:
per_origin = pd.read_csv(ART / "validation_per_origin.csv")
piv = per_origin[per_origin.target == "aggregate_cpu_utilization"].pivot_table(
    index="origin", columns="candidate", values="rmse")
piv[["persistence", "ma24", "drift", "ridge_direct", "gbm_direct"]].round(4)

candidate,persistence,ma24,drift,ridge_direct,gbm_direct
origin,,,,,
60,0.2489,0.2060,0.3041,0.2760,0.2489
66,0.3115,0.2530,0.3662,0.2878,0.3115
72,0.2776,0.3664,0.3090,0.2950,0.2932
78,0.0955,0.3516,0.0974,0.2805,0.0887
84,0.1279,0.2864,0.1212,0.3026,0.1621


In [5]:
print(json.dumps(json.loads((ART / "validation_summary.json").read_text()), indent=2)[:2000])

{
  "n_train": 110,
  "step_seconds": 300.0,
  "folds": [
    60,
    66,
    72,
    78,
    84
  ],
  "horizon": 24,
  "min_train": 60,
  "step": 6,
  "selected": {
    "active_vm_count": "persistence",
    "aggregate_cpu_utilization": "persistence"
  },
  "seed": 0
}


The ranking of persistence, drift and the GBM changes from origin to origin. With
standard error around 0.04 on an RMSE of 0.21, differences below about 0.04 are
noise. That is exactly why the rule broke ties toward simplicity - and exactly why
the holdout result later surprised us.

## Per-horizon behaviour

Errors usually grow with horizon. If they do not, the series is nearly a
random walk and persistence is the right answer.

In [6]:
ph = pd.read_csv(ART / "validation_per_horizon.csv")
cpu = ph[ph.target == "aggregate_cpu_utilization"].set_index("horizon_step")
cpu[["persistence", "ma6", "ma24", "drift", "ridge_direct", "gbm_direct"]].round(4).head(24)

,persistence,ma6,ma24,drift,ridge_direct,gbm_direct
horizon_step,,,,,,
1,0.0884,0.1092,0.2448,0.0881,0.1737,0.1196
2,0.1432,0.1413,0.2522,0.1420,0.2043,0.1525
3,0.1812,0.1996,0.2884,0.1827,0.2302,0.2136
4,0.2263,0.2896,0.3226,0.2335,0.2570,0.2958
5,0.1155,0.0878,0.2263,0.1122,0.1882,0.1071
6,0.1498,0.2063,0.2733,0.1592,0.2108,0.1634
7,0.1977,0.2533,0.2991,0.2101,0.2442,0.2306
8,0.1775,0.2023,0.2605,0.1878,0.1655,0.1754
9,0.2081,0.2657,0.2771,0.2280,0.2113,0.2074


## Operational takeaway

The honest deliverable on 110 points of a nearly unpredictable series is a strong
baseline plus calibrated uncertainty. Complexity buys nothing here, and the
validation harness is what proves it rather than the author's opinion.